# 01-Yue: 数据读入 + 质量控制

In [ ]:
# === PARAMS ===
MANIFEST_PATH = "data/yue/manifest.yaml"
OUTPUT_PATH = "results/01_yue_v1.h5ad"
QC_STRATEGY = "adaptive"
N_MAD = 4
RANDOM_SEED = 42
SOUPX_ENABLED = False
EXPECTED_DOUBLET_RATE = 0.04
FLAG_HEMOGLOBIN = False
FLAG_STRESS_GENES = True
SCORE_CELL_CYCLE = True
DOUBLET_SCORE_THRESHOLD = None  # None=使用 scrublet 自动阈值 | float（如 0.25）=手动覆盖
OUTPUT_VERSION = 1

PER_SAMPLE_MAD = True   # True=按 sample_id 独立算 MAD（推荐），False=全局 MAD（兼容旧行为）
MIN_CELLS_PER_GENE = 3   # 在少于此数量细胞中检测到的基因视为噪声，过滤掉

# --- 固定阈值（仅 QC_STRATEGY="fixed" 时生效）---
MIN_GENES  = 200  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位
MAX_GENES  = 6000  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位
MIN_COUNTS = 500  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位
MAX_PCT_MT = 20  # TODO: 仅 QC_STRATEGY="fixed" 时生效，当前 adaptive 模式下仅占位

In [ ]:
# === Setup：sys.path + 导入依赖 ===
import sys, os
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import shutil
import warnings
import gc
from pathlib import Path
from scipy.stats import median_abs_deviation

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

from scrna_integration.io import sync_gene_ids
from scrna_integration.platform import check_r_available
RSCRIPT_BIN, R_AVAILABLE = check_r_available()


In [ ]:
# 数据读入：txt.gz 格式（Yue organoid，每文件一个细胞，tab 分隔）
# 替代原来的 read_with_manifest / 按透明性铁律，数据读取逻辑拆回 cell
import yaml, anndata
from pathlib import Path
from scrna_integration.io import sync_gene_ids

with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)

data_dir = Path(manifest["input"]["path"])
assert data_dir.exists(), f"数据目录不存在: {data_dir}"

# 读取所有 txt.gz 文件（每个文件 = 一个细胞，行 = 基因，列 = 单个计数）
files = sorted(data_dir.glob("*.txt.gz"))
print(f"发现 {len(files)} 个 txt.gz 文件")

cell_dfs = []
for fp in files:
    df = pd.read_csv(fp, sep="\t", index_col=0, compression="gzip")
    cell_dfs.append(df)

# 列拼接（行=基因，列=细胞）→ 转置为 AnnData（细胞×基因）
combined = pd.concat(cell_dfs, axis=1)
adata = anndata.AnnData(X=sp.csr_matrix(combined.T.values))
adata.var_names = combined.index.tolist()
adata.obs_names = [fp.stem.replace(".txt", "") for fp in files]

# 基础字段
adata.obs["source_dataset"] = manifest["source_dataset"]

# 基因 ID 同步：var.index 为 symbol → 补充 var["ensembl_id"] 列
sync_gene_ids(adata, gene_id_format="symbol")

print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"var 列: {list(adata.var.columns)}")


## 样本级 QC 摘要

按 sample_id 分组统计每个样本的细胞数、基因中位数、UMI 中位数、线粒体比例中位数。
**看什么**：是否存在某个样本与其他样本差异过大（如某样本细胞数极少、或 MT% 异常偏高）。
这将帮助判断是否需要为特定样本设置差异化阈值。

In [ ]:
# 样本级 QC 摘要表
print("===== 样本级 QC 摘要 =====")
sample_summary = adata.obs.groupby("sample_id").agg(
    n_cells=("n_genes", "count"),
    median_n_genes=("n_genes", "median"),
    median_total_counts=("total_counts", "median"),
    median_pct_mt=("pct_counts_mt", "median"),
).sort_values("n_cells", ascending=False)
display(sample_summary)

abnormal = sample_summary[
    (sample_summary["n_cells"] < 100) | (sample_summary["median_pct_mt"] > 30)
]
if len(abnormal) > 0:
    print("\nWARNING 异常样本（n_cells<100 或 median_pct_mt>30%）：")
    display(abnormal)
else:
    print("\n所有样本通过初步检查。")


## 基线 QC 分布

绘制三个主 QC 指标的小提琴图和散点图，供 PI 在设定过滤阈值前直观判断数据质量。

**三个指标的含义**：
- `n_genes`：每个细胞检测到的基因数。过低→空液滴或死细胞；过高→可能是双细胞
- `total_counts`：每个细胞的总 UMI 计数。分布应与 n_genes 正相关
- `pct_counts_mt`：线粒体转录本百分比。过高（>20%）→细胞膜破损/凋亡

**如何使用这些图**：
1. 先看小提琴图，了解各指标的总体分布范围和离群情况
2. 再看散点图，检查 n_genes vs pct_mt 的关系——通常呈负相关
3. 根据分布特征，回到顶部 PARAMS 调整 N_MAD 或固定阈值

In [ ]:
# QC 小提琴图：按 sample_id 分组展示三个主指标。
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="sample_id", rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤前）")
plt.tight_layout()
fig.savefig("results/figures/01_yue_qc_violin_pre.png", dpi=150, bbox_inches="tight")
plt.show()

# QC 散点图
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt", ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤前）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts", ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤前）")
plt.tight_layout()
fig.savefig("results/figures/01_yue_qc_scatter_pre.png", dpi=150, bbox_inches="tight")
plt.show()


## 自适应阈值计算（MAD-based）

MAD（median absolute deviation）是比标准差更稳健的离散度度量，对离群值不敏感。

**参数含义**：
- `N_MAD = 4`：阈值 = 中位数 +/- N_MAD * MAD
- 越大越宽松（保留更多细胞），越小越严格（去除更多细胞）
- `n_genes` 做双侧过滤（过低+过高），`total_counts` 仅下界，`pct_counts_mt` 仅上界

**Why MAD 而不是固定阈值**：不同数据集/样本的 baseline 差异很大（如组织活检 vs 类器官的 MT% 基线差异可达 3-5 倍）。
MAD 基于每个数据集自身的分布自适应调整，避免用一个固定阈值削足适履。

In [ ]:
# 自适应阈值计算（MAD-based）
n_before = adata.n_obs

# 过滤前统计摘要
qc_pre_stats = {
    "n_genes": {"median": float(adata.obs["n_genes"].median()), "mean": float(adata.obs["n_genes"].mean())},
    "total_counts": {"median": float(adata.obs["total_counts"].median()), "mean": float(adata.obs["total_counts"].mean())},
    "pct_counts_mt": {"median": float(adata.obs["pct_counts_mt"].median()), "mean": float(adata.obs["pct_counts_mt"].mean())},
}

if QC_STRATEGY == "adaptive":
    thresholds = {}
    for metric, direction in [("n_genes", "both"), ("total_counts", "lower"), ("pct_counts_mt", "upper")]:
        if PER_SAMPLE_MAD:
            # 按 sample_id 分组，每个样本独立计算阈值
            sample_thresholds = {}
            for sample_id in adata.obs["sample_id"].unique():
                mask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[mask, metric].dropna()
                if len(vals) < 10:
                    print(f"  WARNING: {sample_id} 仅有 {len(vals)} 个细胞的 {metric} 值，沿用全局阈值")
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                lower = max(0, med - N_MAD * mad_val) if direction in ("both", "lower") else None
                upper = med + N_MAD * mad_val if direction in ("both", "upper") else None
                sample_thresholds[sample_id] = {"median": round(med, 1), "mad": round(mad_val, 1),
                                                 "lower": round(lower, 1) if lower else None,
                                                 "upper": round(upper, 1) if upper else None}
            thresholds[metric] = {"mode": "per_sample", "per_sample": sample_thresholds}
        else:
            # 全局 MAD（原逻辑）
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad = median_abs_deviation(vals, nan_policy="omit")
            lower = max(0, med - N_MAD * mad) if direction in ("both", "lower") else None
            upper = med + N_MAD * mad if direction in ("both", "upper") else None
            thresholds[metric] = {"median": round(med, 1), "mad": round(mad, 1),
                                  "lower": round(lower, 1) if lower else None,
                                  "upper": round(upper, 1) if upper else None}
elif QC_STRATEGY == "fixed":
    # 固定阈值模式：适用于已知阈值的数据集（如类器官、实验室内部标准）
    thresholds = {
        "n_genes": {"median": None, "mad": None, "lower": MIN_GENES if 'MIN_GENES' in dir() else 200, "upper": MAX_GENES if 'MAX_GENES' in dir() else 6000},
        "total_counts": {"median": None, "mad": None, "lower": MIN_COUNTS if 'MIN_COUNTS' in dir() else 500, "upper": None},
        "pct_counts_mt": {"median": None, "mad": None, "lower": None, "upper": MAX_PCT_MT if 'MAX_PCT_MT' in dir() else 20},
    }
    print("使用固定阈值模式")
else:
    raise ValueError(f"不支持的 QC_STRATEGY: {QC_STRATEGY}，请使用 'adaptive' 或 'fixed'")

print(f"===== QC 阈值（{QC_STRATEGY}）=====")
print(f"  策略: {QC_STRATEGY}")
if QC_STRATEGY == "adaptive":
    print(f"  N_MAD: {N_MAD}")
    if PER_SAMPLE_MAD:
        print(f"  模式: per-sample（每个 sample_id 独立计算 MAD）")
    else:
        print(f"  模式: global（全局 MAD）")

if PER_SAMPLE_MAD:
    # 打印每个 sample 的阈值汇总表
    print("\n===== 各样本阈值明细 =====")
    for metric in ["n_genes", "total_counts", "pct_counts_mt"]:
        print(f"\n--- {metric} ---")
        rows = []
        for sid, t in thresholds[metric]["per_sample"].items():
            rows.append({"sample_id": sid, **{k: v for k, v in t.items() if v is not None}})
        if rows:
            display(pd.DataFrame(rows).set_index("sample_id"))
else:
    thresh_df = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if vv is not None} for k, v in thresholds.items()}).T
    display(thresh_df)

In [ ]:
# --- 跨样本阈值 forest plot（仅 per-sample 模式）---
# 展示各样本的 MAD 阈值范围，可直观比较各样本的 QC 特性差异
if PER_SAMPLE_MAD and QC_STRATEGY == "adaptive":
    fig, axes = plt.subplots(1, 3, figsize=(15, max(4, len(adata.obs["sample_id"].unique()) * 0.4)))
    for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
        ax = axes[i]
        per_sample = thresholds[metric].get("per_sample", {})
        samples = sorted(per_sample.keys())
        if not samples:
            ax.set_title(f"{metric}\n（无 per-sample 数据）")
            continue
        y_pos = range(len(samples))
        lowers = [per_sample[s].get("lower", 0) or 0 for s in samples]
        uppers = [per_sample[s].get("upper", 0) or 0 for s in samples]
        medians = [per_sample[s].get("median", 0) for s in samples]
        ax.barh(y_pos, [u - l for u, l in zip(uppers, lowers)], left=lowers, height=0.6,
                alpha=0.3, color="steelblue")
        ax.scatter(medians, y_pos, color="red", zorder=5, s=20, label="median")
        ax.set_yticks(y_pos)
        ax.set_yticklabels(samples, fontsize=8)
        ax.set_xlabel(metric)
        ax.set_title(f"{metric} 阈值范围")
        if i == 0:
            ax.legend(fontsize=7, loc="lower right")
    plt.suptitle("Per-sample MAD 阈值 Forest Plot", fontsize=12)
    plt.tight_layout()
    plt.savefig("results/figures/01_yue_qc_forest.png", dpi=150, bbox_inches="tight")
    plt.show()
elif PER_SAMPLE_MAD and QC_STRATEGY == "fixed":
    print("fixed 模式下无 per-sample 阈值，跳过 forest plot")


## N_MAD 敏感度分析

核心问题：N_MAD 太小 -> 丢太多细胞（可能丢真信号）；太大 -> 保留垃圾。
经验法则：组织活检 3-4，类器官 5-7。下面的曲线帮助你选择最佳值。


In [ ]:
# === N_MAD 敏感度分析：帮助 PI 选择最优阈值 ===
# F4修复：敏感度曲线与过滤同口径——PER_SAMPLE_MAD 开关同时作用于曲线与过滤
# 此前曲线始终使用全局 MAD，PI 据曲线选参数会被误导
_test_mads = [3, 4, 5, 6, 7]
sensitivity_results = []
for _nm in _test_mads:
    _keep = pd.Series(True, index=adata.obs_names)
    for metric, direction in [("n_genes", "both"), ("total_counts", "lower"), ("pct_counts_mt", "upper")]:
        if PER_SAMPLE_MAD:
            # 按 sample_id 分组独立算 MAD（与阈值计算 + 实际过滤同口径）
            for sample_id in adata.obs["sample_id"].unique():
                smask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[smask, metric].dropna()
                # 样本细胞数不足 10，无法可靠估计 MAD，该样本细胞不参与本维度过滤
                if len(vals) < 10:
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                if direction in ("both", "lower"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] >= max(0, med - _nm * mad_val)
                if direction in ("both", "upper"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] <= med + _nm * mad_val
        else:
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad_val = median_abs_deviation(vals, nan_policy="omit")
            if direction in ("both", "lower"):
                _keep &= adata.obs[metric] >= max(0, med - _nm * mad_val)
            if direction in ("both", "upper"):
                _keep &= adata.obs[metric] <= med + _nm * mad_val
    n_keep = _keep.sum()
    sensitivity_results.append({
        "N_MAD": _nm,
        "cells_kept": n_keep,
        "pct_kept": round(100 * n_keep / adata.n_obs, 1),
        "median_mt_kept": round(adata.obs.loc[_keep, "pct_counts_mt"].median(), 2),
    })

sens_df = pd.DataFrame(sensitivity_results)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sens_df["N_MAD"], sens_df["pct_kept"], "o-", color="steelblue", linewidth=2)
ax1.axvline(N_MAD, color="red", linestyle="--", label=f"当前 N_MAD={N_MAD}")
ax1.set_xlabel("N_MAD")
ax1.set_ylabel("细胞保留率 (%)", color="steelblue")
ax1.set_title("N_MAD 敏感度：保留率 vs 阈值宽松度")
ax2 = ax1.twinx()
ax2.plot(sens_df["N_MAD"], sens_df["median_mt_kept"], "s--", color="orange")
ax2.set_ylabel("保留细胞的 median MT%", color="orange")
ax1.legend()
plt.tight_layout()
plt.savefig("results/figures/01_yue_mad_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print(sens_df.to_string(index=False))
print("\n经验判据：保留 85-95% 细胞 + median MT% 不显著上升 = 合理")


## 基因复杂度

`log_complexity = log10(n_genes+1) / log10(total_counts+1)` 反映每个细胞的"基因多样性密度"。
复杂度异常低（相同 UMI 下基因数过少）提示该细胞可能只捕获了极少数高表达基因，
是低质量细胞的补充判据。

In [ ]:
# 基因复杂度
adata.obs["log_complexity"] = np.log10(adata.obs["n_genes"] + 1) / np.log10(adata.obs["total_counts"] + 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adata.obs["log_complexity"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("log10(n_genes+1) / log10(total_counts+1)")
ax.set_ylabel("细胞数")
ax.set_title("基因复杂度分布")
for pct, color in [(1, "red"), (5, "orange"), (50, "green"), (95, "orange"), (99, "red")]:
    val = np.percentile(adata.obs["log_complexity"].dropna(), pct)
    ax.axvline(val, color=color, linestyle="--", alpha=0.5, linewidth=0.8)
plt.tight_layout()
fig.savefig("results/figures/01_yue_complexity.png", dpi=150, bbox_inches="tight")
plt.show()
for pct in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  P{pct}: {np.percentile(adata.obs['log_complexity'].dropna(), pct):.4f}")


## 特殊基因标记

**血红蛋白基因（HB）**：红细胞污染/溶解的标志。仅标记不移除——消化道活检中低水平 HB 背景常见。
**应激基因**：即时早期基因（IEGs）+ 热休克蛋白（HSPs），在组织解离过程中被机械/酶切应激诱导。
标记供下游分析参考，不是过滤依据。

In [ ]:
# FLAG_HEMOGLOBIN=False，跳过血红蛋白标记（类器官数据集无 RBC 污染）。

# 应激基因标记（仅标记，不移除）
if FLAG_STRESS_GENES:
    STRESS_GENES = ['JUN', 'FOS', 'JUNB', 'FOSB', 'ATF3', 'HSPA1A', 'HSPA1B', 'HSP90AA1', 'HSP90AB1', 'DNAJB1', 'HSPB1']
    stress_in_data = [g for g in STRESS_GENES if g in adata.var_names]
    if stress_in_data:
        adata.var["stress"] = adata.var_names.isin(stress_in_data)
        sc.pp.calculate_qc_metrics(adata, qc_vars=["stress"], percent_top=None, log1p=False, inplace=True)
        print(f"应激基因: {len(stress_in_data)}/{len(STRESS_GENES)} 个检测到, 中位 pct={adata.obs['pct_counts_stress'].median():.2f}%")
    else:
        print("WARNING: 数据中未检测到应激基因")
else:
    print("⏭ 跳过应激基因标记（FLAG_STRESS_GENES=False）")

## 双细胞鉴定（Scrublet）

双细胞（doublet）是两个细胞被误包在同一个液滴中测序。其基因表达是两种细胞类型的混合，
会干扰细胞类型注释和差异表达分析。

**处理策略**：使用 manifest-driven 跳过逻辑。
- 若 manifest `preprocessing_done` 含 `doublet_removal`→跳过（作者已处理）
- 若 manifest `qc_overrides.doublet_removal.skip=true`→跳过
- 否则按样本独立运行 scrublet

**注意**：doublet 仅标记不移除。PI 在查看下游聚类后可根据 doublet 是否形成独立小群来决定。

In [ ]:
# 双细胞鉴定：manifest-driven skip check + per-sample scrublet
import scrublet as scr
import yaml

if "doublet_score" not in adata.obs.columns:
    adata.obs["doublet_score"] = np.nan
    adata.obs["predicted_doublet"] = False

with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)
pp_done = manifest.get("preprocessing_done", [])
qc_override = manifest.get("qc_overrides", {}).get("doublet_removal", {})

skip_doublet = False
skip_reason = None
if "doublet_removal" in pp_done:
    skip_doublet = True
    skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
elif qc_override.get("skip"):
    skip_doublet = True
    skip_reason = qc_override.get("reason", "qc_overrides.doublet_removal.skip=True")

if skip_doublet:
    print(f"双细胞鉴定已跳过: {skip_reason}")
    adata.obs["doublet_score"] = np.nan
    adata.obs["predicted_doublet"] = False
elif EXPECTED_DOUBLET_RATE is None:
    print("EXPECTED_DOUBLET_RATE=None，跳过 scrublet")
    adata.obs["doublet_score"] = np.nan
    adata.obs["predicted_doublet"] = False
else:
    print(f"运行 Scrublet per sample (expected_doublet_rate={EXPECTED_DOUBLET_RATE})...")
    for sample_id in sorted(adata.obs["sample_id"].unique()):
        sample_mask = adata.obs["sample_id"] == sample_id
        n_sample = sample_mask.sum()
        if n_sample < 50:
            print(f"  {sample_id}: {n_sample} 细胞（太少），跳过")
            continue
        sub = adata[sample_mask].copy()
        scrub = scr.Scrublet(sub.X, expected_doublet_rate=EXPECTED_DOUBLET_RATE, random_state=RANDOM_SEED)
        doublet_scores, predicted_doublets = scrub.scrub_doublets()
        # 可视化 doublet score 分布（判断双峰性）——每个 sample 独立出图
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(doublet_scores, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
        auto_threshold = scrub.threshold_
        ax.axvline(auto_threshold, color="red", linestyle="--", label=f"自动阈值={auto_threshold:.3f}")
        if DOUBLET_SCORE_THRESHOLD is not None:
            ax.axvline(DOUBLET_SCORE_THRESHOLD, color="orange", linestyle="-", linewidth=2, label=f"手动阈值={DOUBLET_SCORE_THRESHOLD}")
            # 用手动阈值覆盖自动阈值结果
            predicted_doublets = doublet_scores > DOUBLET_SCORE_THRESHOLD
        ax.set_xlabel("Doublet Score")
        ax.set_ylabel("细胞数")
        ax.set_title(f"{sample_id}: Doublet Score 分布")
        ax.legend()
        plt.tight_layout(); plt.show()
        adata.obs.loc[sample_mask, "doublet_score"] = doublet_scores
        adata.obs.loc[sample_mask, "predicted_doublet"] = predicted_doublets
        n_dbl = int(predicted_doublets.sum())
        print(f"  {sample_id}: {n_dbl}/{n_sample} predicted doublets ({100*n_dbl/n_sample:.1f}%)")

n_doublets_total = int(adata.obs["predicted_doublet"].sum())
pct_doublets = 100 * n_doublets_total / adata.n_obs if adata.n_obs > 0 else 0
print(f"\n双细胞汇总: {n_doublets_total}/{adata.n_obs} predicted doublets ({pct_doublets:.1f}%)")


## 环境 RNA 校正（SoupX）

环境 RNA（ambient RNA）来自裂解的细胞碎片和游离 RNA，悬浮在液滴溶液中并被随机捕获形成背景噪声。
SoupX 利用 raw matrix（含空液滴/碎片背景）和 filtered matrix（只含真实细胞）之间的差异，
估计每个基因的污染比例并扣除。

**为什么用 subprocess Rscript 而不是 rpy2**：
rpy2 + anndata2ri 在 conda R 4.4.3 下存在严重兼容性问题。subprocess 独立进程通过临时 mtx 文件交换数据，进程隔离避免桥接崩溃。

**三重守卫**（任一不满足则优雅跳过）：
1. `SOUPX_ENABLED=True`
2. Manifest 声明了 `raw_matrix_path`
3. Rscript 可执行 + SoupX R 包可加载

In [ ]:
# 环境 RNA 校正（SoupX）—— subprocess Rscript 模式
soupx_applied = False
n_soupx_corrected = 0

if not SOUPX_ENABLED:
    print(f"SoupX 已跳过: SOUPX_ENABLED=False（类器官数据集无 raw matrix）。")
elif not adata.uns.get("raw_matrix_path"):
    print("SoupX 已跳过: adata.uns 中无 raw_matrix_path。")
elif not R_AVAILABLE:
    print("SoupX 已跳过: R 环境未就绪。")
else:
    raw_path = adata.uns["raw_matrix_path"]
    soupx_script = "scripts/soupx_run.R"
    soupx_tmp = "results/_soupx_tmp"
    os.makedirs(soupx_tmp, exist_ok=True)
    import scipy.io

    print(f"raw_matrix_path: {raw_path}")
    if "ambient_correction_applied" not in adata.obs.columns:
        adata.obs["ambient_correction_applied"] = False

    for sample_id in sorted(adata.obs["sample_id"].unique()):
        sample_mask = adata.obs["sample_id"] == sample_id
        n_cells = sample_mask.sum()
        cell_ids = adata.obs_names[sample_mask]

        # 提取原始 10x barcode
        prefix = f"{sample_id}_"
        original_barcodes = []
        for cid in cell_ids:
            if not cid.startswith(prefix):
                warnings.warn(f"cell_id '{cid}' barcode 解析异常")
                original_barcodes.append(cid)
                continue
            rest = cid[len(prefix):]
            parts = rest.rsplit("-", 1)
            original_barcodes.append(parts[0] if len(parts) == 2 and parts[1].isdigit() else rest)

        # 定位 raw matrix
        raw_sample_dir = Path(raw_path) / sample_id / "raw_feature_bc_matrix"
        if not (raw_sample_dir / "matrix.mtx.gz").exists() and not (raw_sample_dir / "matrix.mtx").exists():
            alt_raw = Path(raw_path) / "raw_feature_bc_matrix"
            if (alt_raw / "matrix.mtx.gz").exists() or (alt_raw / "matrix.mtx").exists():
                raw_sample_dir = alt_raw
            else:
                print(f"  {sample_id}: raw_feature_bc_matrix 未找到，跳过")
                continue

        try:
            sub_adata = adata[cell_ids].copy()
            filtered_export = os.path.join(soupx_tmp, f"{sample_id}_filtered")
            os.makedirs(filtered_export, exist_ok=True)
            count_mtx = sp.csr_matrix(sub_adata.X).T
            scipy.io.mmwrite(os.path.join(filtered_export, "matrix.mtx"), count_mtx)
            with open(os.path.join(filtered_export, "barcodes.tsv"), "w") as f:
                f.write("\n".join(original_barcodes) + "\n")
            with open(os.path.join(filtered_export, "features.tsv"), "w") as f:
                for gn in sub_adata.var_names:
                    f.write(f"{gn}\t{gn}\tGene Expression\n")

            work_dir = os.path.join(soupx_tmp, sample_id)
            cmd = [RSCRIPT_BIN, "--vanilla", soupx_script,
                   os.path.abspath(work_dir), os.path.abspath(filtered_export),
                   os.path.abspath(str(raw_sample_dir)), sample_id]
            print(f"  {sample_id}: 执行 SoupX...")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
            for line in result.stdout.strip().split("\n"):
                print(f"    [R] {line}")
            if result.returncode != 0:
                print(f"  Rscript 失败 (exit={result.returncode}): {result.stderr[:300]}")
                continue

            corrected_mtx_fp = os.path.join(work_dir, "corrected_counts.mtx")
            if not os.path.exists(corrected_mtx_fp):
                print(f"  校正矩阵未产出: {corrected_mtx_fp}")
                continue
            corrected = sp.csr_matrix(scipy.io.mmread(corrected_mtx_fp).T)
            if corrected.shape != (n_cells, sub_adata.n_vars):
                print(f"  形状不匹配，跳过")
                continue

            # barcode + gene order validation
            with open(os.path.join(work_dir, "barcodes.tsv")) as f:
                if [l.strip() for l in f] != original_barcodes:
                    print(f"  barcode 顺序不匹配，跳过")
                    continue
            out_features = pd.read_csv(os.path.join(work_dir, "features.tsv"), sep="\t", header=None)
            if out_features.iloc[:, 1].tolist() != sub_adata.var_names.tolist():
                print(f"  基因顺序不匹配，跳过")
                continue

            # F1修复：对布尔/花式索引产生的 AnnData 视图赋值不会写回原对象
            # scanpy 只弹 warning 而不实际修改底层 CSR 矩阵中的值
            # 改为整数位置索引直接写 CSR 矩阵，并做校正前后差异断言防止静默失败
            cell_indices = np.where(sample_mask.values)[0]
            orig_sum = adata.X[cell_indices, :].sum()
            adata.X[cell_indices, :] = corrected
            corrected_sum = adata.X[cell_indices, :].sum()
            max_diff = abs(corrected_sum - orig_sum)
            if max_diff > 0:
                adata.obs.loc[cell_ids, "ambient_correction_applied"] = True
                n_soupx_corrected += n_cells
                print(f"  {sample_id}: SoupX 完成, {n_cells} 细胞已校正, max_diff={max_diff:.2e}")
            else:
                print(f"  WARNING {sample_id}: SoupX 校正后矩阵无差异（max_diff={max_diff}），可能原始数据未被 R 端修改，跳过标记")
            del sub_adata
        except subprocess.TimeoutExpired:
            print(f"  {sample_id}: 超时（10min），跳过")
        except Exception as e:
            print(f"  {sample_id}: 异常 ({type(e).__name__}): {e}")

    if n_soupx_corrected > 0:
        soupx_applied = True
    print(f"\nSoupX 校正: {n_soupx_corrected} 个细胞已校正")
    print(f"ambient_correction_applied: {adata.obs['ambient_correction_applied'].value_counts().to_dict()}")
    shutil.rmtree(soupx_tmp, ignore_errors=True)


## 细胞周期评分

使用 Tirosh et al. (2015) 的 S 期和 G2M 期 marker genes 对每个细胞打分。
细胞周期阶段（G1/S/G2M）是重要的技术协变量——如果不同样本/条件的细胞周期分布
不均衡，可能在差异表达分析中引入混淆。

In [ ]:
# 细胞周期评分（Tirosh 2015 marker genes）
if SCORE_CELL_CYCLE:
    s_genes = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
    g2m_genes = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
    print("细胞周期评分完成")
    print(adata.obs["phase"].value_counts())
else:
    print("SCORE_CELL_CYCLE=False，跳过")


## 过滤

根据上一步得出的阈值过滤低质量细胞。

**不在此步移除的**：
- 双细胞（predicted_doublet）：仅标记，供下游聚类后决定
- 血红蛋白高表达细胞：仅标记（flag_hb）
- 应激高表达细胞：仅标记

In [ ]:
# 过滤：应用 QC 阈值
print("===== QC 过滤 =====")
cells_before = adata.n_obs
print(f"过滤前细胞数: {cells_before:,}")

if PER_SAMPLE_MAD:
    # Per-sample 模式：按每个 sample 的阈值独立判断
    keep = pd.Series(True, index=adata.obs_names)
    filter_counts = {"n_genes_lower": 0, "n_genes_upper": 0, "total_counts": 0, "pct_counts_mt": 0}
    for sample_id in adata.obs["sample_id"].unique():
        smask = adata.obs["sample_id"] == sample_id
        st = thresholds["n_genes"]["per_sample"].get(sample_id)
        if st and st.get("lower") is not None:
            n_fail = ((adata.obs.loc[smask, "n_genes"] < st["lower"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "n_genes"] >= st["lower"]
            filter_counts["n_genes_lower"] += n_fail
        if st and st.get("upper") is not None:
            n_fail = ((adata.obs.loc[smask, "n_genes"] > st["upper"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "n_genes"] <= st["upper"]
            filter_counts["n_genes_upper"] += n_fail
        st_tc = thresholds["total_counts"]["per_sample"].get(sample_id)
        if st_tc and st_tc.get("lower") is not None:
            n_fail = ((adata.obs.loc[smask, "total_counts"] < st_tc["lower"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "total_counts"] >= st_tc["lower"]
            filter_counts["total_counts"] += n_fail
        st_mt = thresholds["pct_counts_mt"]["per_sample"].get(sample_id)
        if st_mt and st_mt.get("upper") is not None:
            n_fail = ((adata.obs.loc[smask, "pct_counts_mt"] > st_mt["upper"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "pct_counts_mt"] <= st_mt["upper"]
            filter_counts["pct_counts_mt"] += n_fail
    print(f"  n_genes 偏低: {filter_counts['n_genes_lower']:,}  偏高: {filter_counts['n_genes_upper']:,}")
    print(f"  total_counts 偏低: {filter_counts['total_counts']:,}")
    print(f"  pct_counts_mt 偏高: {filter_counts['pct_counts_mt']:,}")
else:
    keep = pd.Series(True, index=adata.obs_names)
    if thresholds["n_genes"]["lower"] is not None:
        n_fail = (adata.obs["n_genes"] < thresholds["n_genes"]["lower"]).sum()
        keep &= adata.obs["n_genes"] >= thresholds["n_genes"]["lower"]
        print(f"  n_genes < {thresholds['n_genes']['lower']:.0f}: {n_fail:,} 失败")
    if thresholds["n_genes"]["upper"] is not None:
        n_fail = (adata.obs["n_genes"] > thresholds["n_genes"]["upper"]).sum()
        keep &= adata.obs["n_genes"] <= thresholds["n_genes"]["upper"]
        print(f"  n_genes > {thresholds['n_genes']['upper']:.0f}: {n_fail:,} 失败")
    if thresholds["total_counts"]["lower"] is not None:
        n_fail = (adata.obs["total_counts"] < thresholds["total_counts"]["lower"]).sum()
        keep &= adata.obs["total_counts"] >= thresholds["total_counts"]["lower"]
        print(f"  total_counts < {thresholds['total_counts']['lower']:.0f}: {n_fail:,} 失败")
    if thresholds["pct_counts_mt"]["upper"] is not None:
        n_fail = (adata.obs["pct_counts_mt"] > thresholds["pct_counts_mt"]["upper"]).sum()
        keep &= adata.obs["pct_counts_mt"] <= thresholds["pct_counts_mt"]["upper"]
        print(f"  pct_counts_mt > {thresholds['pct_counts_mt']['upper']:.1f}%: {n_fail:,} 失败")

print(f"\n保留细胞: {keep.sum():,} / {cells_before:,} ({100*keep.sum()/cells_before:.1f}%)")

obs_pre_filter = adata.obs.copy()  # åå­å®å¨ï¼åªå¤å¶ obs DataFrameï¼ä¸å¤å¶ç©éµ
adata = adata[keep].copy()
cells_after = adata.n_obs
print(f"去除细胞数: {cells_before - cells_after:,} ({100*(cells_before-cells_after)/cells_before:.1f}%)")

# 基因过滤：移除仅在极少细胞中检测到的噪声基因
n_genes_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
n_genes_after = adata.n_vars
print(f"基因过滤：{n_genes_before:,} → {n_genes_after:,}（移除 {n_genes_before - n_genes_after:,} 个仅在 <{MIN_CELLS_PER_GENE} 细胞中检测到的基因）")

## 过滤交叉诊断

MAD 过滤与各标记物（doublet/HB/stress）的关系：被过滤掉的细胞是否富集了这些标记？


In [ ]:
# === 过滤交叉诊断：MAD 过滤与各标记物的关系 ===
n_removed_total = cells_before - cells_after
if n_removed_total > 0 and 'obs_pre_filter' in dir():
    print("===== 被过滤细胞的标记物富集分析 =====")
    _removed_idx = obs_pre_filter.index.difference(adata.obs_names)
    _kept_idx = adata.obs_names
    
    enrichment = {}
    if "predicted_doublet" in obs_pre_filter.columns:
        dbl_removed = obs_pre_filter.loc[_removed_idx, "predicted_doublet"].mean()
        dbl_kept = obs_pre_filter.loc[_kept_idx, "predicted_doublet"].mean()
        enrichment["doublet"] = {
            "在被过滤细胞中": f"{dbl_removed:.1%}",
            "在保留细胞中": f"{dbl_kept:.1%}",
            "富集倍数": round(dbl_removed / max(dbl_kept, 0.001), 1)
        }
    if "flag_hb" in obs_pre_filter.columns:
        hb_removed = obs_pre_filter.loc[_removed_idx, "flag_hb"].mean()
        hb_kept = obs_pre_filter.loc[_kept_idx, "flag_hb"].mean()
        enrichment["hemoglobin"] = {
            "在被过滤细胞中": f"{hb_removed:.1%}",
            "在保留细胞中": f"{hb_kept:.1%}",
            "富集倍数": round(hb_removed / max(hb_kept, 0.001), 1)
        }
    if "pct_counts_stress" in obs_pre_filter.columns:
        stress_removed = obs_pre_filter.loc[_removed_idx, "pct_counts_stress"].median()
        stress_kept = obs_pre_filter.loc[_kept_idx, "pct_counts_stress"].median()
        enrichment["stress(median_pct)"] = {
            "在被过滤细胞中": f"{stress_removed:.2f}%",
            "在保留细胞中": f"{stress_kept:.2f}%",
            "富集倍数": round(stress_removed / max(stress_kept, 0.001), 1)
        }
    if "log_complexity" in obs_pre_filter.columns:
        cx_removed = obs_pre_filter.loc[_removed_idx, "log_complexity"].median()
        cx_kept = obs_pre_filter.loc[_kept_idx, "log_complexity"].median()
        enrichment["complexity(median)"] = {
            "在被过滤细胞中": f"{cx_removed:.3f}",
            "在保留细胞中": f"{cx_kept:.3f}",
            "富集倍数": "N/A"
        }
    
    if enrichment:
        display(pd.DataFrame(enrichment).T)
        print("\n解读：富集倍数 > 3 = MAD 过滤已隐含覆盖该标记物；≈ 1 = 两者独立，下游需额外处理")
    
    del obs_pre_filter  # 释放内存
else:
    print("无细胞被过滤，跳过交叉诊断")


## Per-sample 过滤影响


In [ ]:
# === Per-sample 过滤影响表 ===
if PER_SAMPLE_MAD and "filter_per_sample_stats" not in dir():
    pass  # 已通过 per-sample 阈值实现，此处汇总过滤后细胞分布

print("===== Per-sample 过滤影响 =====")
if "qc_report_v1" in adata.uns:
    print(f"总计: {adata.uns['qc_report_v1']['cells_before']:,} -> {adata.uns['qc_report_v1']['cells_after']:,} "
          f"(去除 {adata.uns['qc_report_v1']['pct_removed']}%)")
# 每个 sample 的细胞数（过滤后）
sample_counts = adata.obs["sample_id"].value_counts().sort_index()
print(sample_counts.to_string())
total = sample_counts.sum()
print(f"\n各 sample 占比:")
for sid, n in sample_counts.items():
    print(f"  {sid}: {n:,} ({100*n/total:.1f}%)")
min_sample = sample_counts.idxmin()
max_sample = sample_counts.idxmax()
if sample_counts.max() / max(sample_counts.min(), 1) > 5:
    print(f"\n⚠️ 样本间细胞数差异 > 5 倍（{max_sample}={sample_counts.max()} vs {min_sample}={sample_counts.min()}）")
    print("  -> 下游整合时可能需要考虑下采样平衡（02_merged 的 DOWNSAMPLE_TO_MIN）")


## 过滤前后对比

复刻过滤前的 QC 图，供 PI 做直观对比。

**对比检查要点**：
- 小提琴图：各指标的分布尾部是否被正确截断
- 散点图：被移除的细胞是否集中在预期区域（低基因数 + 高 MT% 区）
- 剩余细胞数是否合理：通常保留 80-95%

In [ ]:
# 过滤后 QC 小提琴图
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="sample_id", rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤后）")
plt.tight_layout()
fig.savefig("results/figures/01_yue_qc_violin_post.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt", ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤后）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts", ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤后）")
plt.tight_layout()
fig.savefig("results/figures/01_yue_qc_scatter_post.png", dpi=150, bbox_inches="tight")
plt.show()


## QC 报告摘要

汇总本次 QC 的全部参数与结果，写入 `adata.uns["qc_report_v1"]`，供下游 notebook 读取。

In [ ]:
# QC 报告摘要
n_removed = cells_before - cells_after
qc_report = {
    "strategy": QC_STRATEGY,
    "n_mad": N_MAD if QC_STRATEGY == "adaptive" else None,
    "thresholds": {k: {kk: vv for kk, vv in v.items() if vv is not None} for k, v in thresholds.items()},
    "cells_before": int(cells_before),
    "cells_after": int(cells_after),
    "cells_removed": int(n_removed),
    "pct_removed": round(100 * n_removed / cells_before, 1) if cells_before > 0 else 0,
    "pre_qc_stats": qc_pre_stats,
    "doublet_rate_pct": round(100 * n_doublets_total / cells_before, 2) if cells_before > 0 else 0,
    "soupx_applied": soupx_applied,
    "soupx_cells_corrected": n_soupx_corrected,
    "cell_cycle_scored": SCORE_CELL_CYCLE,
    "flag_hb": FLAG_HEMOGLOBIN if 'FLAG_HEMOGLOBIN' in dir() else True,
    "n_hb_flagged": n_hb_flagged if 'n_hb_flagged' in dir() else 0,
}
adata.uns["qc_report_v1"] = qc_report
print("===== QC 报告摘要 =====")
for k, v in qc_report.items():
    print(f"  {k}: {v}")


In [ ]:
# Checkpoint：写入 per-dataset h5ad
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, f"adata.X invariant broken: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"

# F3修复：基因 ID 轴统一性断言——检查 per-dataset 内基因名无大小写混用
# 过大写/小写混用在 merge 时会导致 inner join 基因交集意外坍塌
_gene_names = list(adata.var_names)
_upper_count = sum(1 for g in _gene_names if g[0].isupper()) if _gene_names else 0
_lower_count = sum(1 for g in _gene_names if g[0].islower()) if _gene_names else 0
_total = len(_gene_names)
if _upper_count > 0 and _lower_count > 0:
    raise ValueError(
        f"基因 ID 轴不一致：{_upper_count} 个大写首字母基因 + {_lower_count} 个小写首字母基因 共 {_total} 个。"
        f"请统一基因名大小写（如全部 .str.upper()）后再进入 merge。"
    )
print(f"基因 ID 轴一致性检查通过：{_total} 个基因，统一为{'大写' if _upper_count > 0 else '小写'}首字母")

adata.uns["stage"] = "01_qcd"

adata.uns["status"] = "experimental"
adata.uns["upstream"] = [MANIFEST_PATH]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"OK 写入 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")
assert os.path.exists(OUTPUT_PATH), f"输出未找到: {OUTPUT_PATH}"
print(f"已校验: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")
del adata; gc.collect()
print("内存已释放。")
